### Lab 4.1 - Dataset Challenge
Your task in this lab is to set up and train a neural network on any dataset of your choosing.   Look at UCI ML Repo and Kaggle to find datasets, for example.  

Train a neural network on the dataset without any regularization or other special techniques to get a baseline train and test error.  Then see how much you can improve the network's test error through techniques learned in class like regularization, different optimizers, batch normalization, etc.

In [1]:
import numpy as np
import torch

In [2]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17) 
  
# data (as pandas dataframes) 
X = breast_cancer_wisconsin_diagnostic.data.features 
y = breast_cancer_wisconsin_diagnostic.data.targets 
  
# metadata 
# print(breast_cancer_wisconsin_diagnostic.metadata) 
  
# variable information 
# print(breast_cancer_wisconsin_diagnostic.variables) 

In [3]:
# X.columns

In [4]:
bad = X.isna().any(axis=1)
X = X[~bad]
y = y[~bad]

In [5]:
X = X.values.astype('float64')
y = y.values.reshape(-1)

# map 'B' (Benign) to 0, 'M' (Malignant) to 1
y = np.where(y == 'B', 0, 1)
y.shape

(569,)

In [6]:
X -= np.mean(X,axis=0)
X /= np.std(X,axis=0)

In [7]:
print(f'X: {X.shape}')
print(f'y: {y.shape}')

X: (569, 30)
y: (569,)


In [8]:
# examine class imbalance
n = len(y)
percent_malignant = np.sum(y) / n
print(f'Data is {percent_malignant*100:.2f}% malignant, {(1-percent_malignant)*100:.2f}% benign')

Data is 37.26% malignant, 62.74% benign


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.90, random_state=0)

print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_test: {y_test.shape}')

X_train: (512, 30)
y_train: (512,)
X_test: (57, 30)
y_test: (57,)


In [10]:
# examine class imbalance in training/test sets
percent_malignant_train = np.sum(y_train) / len(X_train)
percent_malignant_test = np.sum(y_test) / len(X_test)
print(f'Training Data is {percent_malignant_train*100:.2f}% malignant, {(1-percent_malignant_train)*100:.2f}% benign')
print(f'Test Data is {percent_malignant_test*100:.2f}% malignant, {(1-percent_malignant_test)*100:.2f}% benign')

Training Data is 37.11% malignant, 62.89% benign
Test Data is 38.60% malignant, 61.40% benign


In [11]:
# convert to tensors for use in pytorch
X_train = torch.tensor(X_train).float()
X_test = torch.tensor(X_test).float()
y_train = torch.tensor(y_train).long()
y_test = torch.tensor(y_test).long()

NN Training

In [12]:
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import Sequential, Linear, ReLU

In [13]:
def compute_model_acc(model: torch.nn.Sequential, X: torch.Tensor, y: torch.Tensor) -> float:
    z = model(X)
    y_predict = torch.argmax(z, dim=1)
    num_correct = torch.sum(y_predict == y)
    n = len(y)
    return num_correct / n

In [14]:
train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [15]:
input_size = 30
hidden_size = 60
output_size = 2  # binary classification

model = Sequential(
    Linear(input_size, hidden_size),
    ReLU(),
    Linear(hidden_size, output_size)
)

In [16]:
loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.SGD(model.parameters(), lr=1e-3)

In [17]:
epochs = 100
model.train()
for epoch in range(epochs):
    for X_batch, y_batch in train_loader:
        opt.zero_grad() # zero out the gradients

        z_batch = model(X_batch) # compute z values
        loss = loss_fn(z_batch,y_batch) # compute loss

        loss.backward() # compute gradients

        opt.step() # apply gradients

    print(f'epoch {epoch}: loss is {loss.item():.3f} -- training accuracy is {compute_model_acc(model, X_train, y_train):.3f}, test accuracy is {compute_model_acc(model, X_test, y_test):.3f}')

epoch 0: loss is 0.686 -- training accuracy is 0.551, test accuracy is 0.561
epoch 1: loss is 0.701 -- training accuracy is 0.619, test accuracy is 0.614
epoch 2: loss is 0.736 -- training accuracy is 0.664, test accuracy is 0.702
epoch 3: loss is 0.639 -- training accuracy is 0.713, test accuracy is 0.702
epoch 4: loss is 0.632 -- training accuracy is 0.760, test accuracy is 0.719
epoch 5: loss is 0.572 -- training accuracy is 0.799, test accuracy is 0.719
epoch 6: loss is 0.627 -- training accuracy is 0.826, test accuracy is 0.737
epoch 7: loss is 0.589 -- training accuracy is 0.834, test accuracy is 0.754
epoch 8: loss is 0.561 -- training accuracy is 0.840, test accuracy is 0.772
epoch 9: loss is 0.578 -- training accuracy is 0.854, test accuracy is 0.807
epoch 10: loss is 0.567 -- training accuracy is 0.863, test accuracy is 0.825
epoch 11: loss is 0.596 -- training accuracy is 0.875, test accuracy is 0.825
epoch 12: loss is 0.521 -- training accuracy is 0.887, test accuracy is 0.

In [18]:
model.eval()
print(f'Model final test accuracy: {compute_model_acc(model, X_test, y_test)*100:.2f}%')

Model final test accuracy: 92.98%
